# 01 — Data Explore (L0 stub)

Scaffold for L0/L1 data viz. Teammate lane (non-blocking) per plan Sec 7.

- Visualize `data/*/metadata.csv` (image_path, lat, lon)
- Plot cell distribution, country stratification, image samples
- Check for spatial bias / balance before L1 training

Run after `scripts/download_subset.py --dry-run` or real download.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Windows-safe paths
ROOT = Path.cwd()
# If notebook is in notebooks/, project root is parent
if (ROOT / "notebooks" / "01_data_explore.ipynb").exists():
    pass  # running from project root
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

print(f"ROOT={ROOT}")
print(f"src exists: {(ROOT / 'src').exists()}")
try:
    import src.dataset, src.cells
    print("src imports ok")
except Exception as e:
    print(f"src import failed: {e}")
    import sys; sys.path.insert(0, str(ROOT))
    import src.dataset, src.cells
    print("src imports ok (after path fix)")

In [ ]:
# Load a metadata.csv (dummy from --dry-run or real)
csv_path = ROOT / "data" / "flickr_geo_tiny" / "metadata.csv"
if not csv_path.exists():
    print(f"No CSV yet at {csv_path} — run:")
    print("  python scripts/download_subset.py --dataset flickr_geo_tiny --output-dir data/flickr_geo_tiny --max-samples 50 --dry-run")
else:
    df = pd.read_csv(csv_path)
    print(df.head())
    print(f"rows={len(df)} columns={list(df.columns)}")
    # Quick map
    if {"lat","lon"}.issubset(df.columns):
        plt.figure()
        plt.scatter(df["lon"], df["lat"], s=4, alpha=0.6)
        plt.xlabel("lon"); plt.ylabel("lat"); plt.title("Samples (lon vs lat)")
        plt.show()
        print(df["lat"].describe())
        print(df["lon"].describe())

In [ ]:
# Cells stub — quad-tree K~300 (L1 will replace with density-driven splits)
from src.cells import build_cells, assign_cells, cells_to_centroids
import numpy as np

if 'df' in locals():
    coords = df[["lat","lon"]].values
    cells = build_cells(coords, K=300, method="quad_tree")
    print(f"cells={len(cells)} (stub grid)")
    # Balance check (images per cell)
    counts = [c.count for c in cells]
    print(f"per-cell count: min={min(counts)} max={max(counts)} mean={np.mean(counts):.1f} std={np.std(counts):.1f}")
    # Assign + histogram
    ids = assign_cells(coords, cells)
    plt.figure()
    plt.hist(ids, bins=min(30, len(cells)))
    plt.xlabel("cell_id"); plt.ylabel("count"); plt.title("Images per cell (stub)")
    plt.show()
else:
    print("No df — run previous cell first")

In [ ]:
# Dataset + transforms smoke (224)
from src.dataset import GeoDataset, get_transforms
from PIL import Image

if 'csv_path' in locals() and csv_path.exists() and len(df) > 0:
    ds = GeoDataset(csv_path, image_size=224, train=True)
    img, label = ds[0]
    print(f"image tensor: {img.shape} dtype={img.dtype}  label={label}")
    # Visualize one augmented sample
    inv_norm = lambda t: t * 0.229 + 0.485  # approx — per-channel would be better
    plt.figure()
    plt.imshow(img.permute(1,2,0).numpy().clip(0,1))
    plt.title(f"sample 0 — label {label}"); plt.axis("off"); plt.show()
else:
    print("No dataset available")